In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from datasets import Dataset

ROOT = Path.cwd().resolve().parent
if str(ROOT / "src") not in sys.path:
    sys.path.append(str(ROOT / "src"))

from project_paths import get_paths
from imdb_spoiler_io import load_raw_imdb_spoiler_json, prepare_reviews_dataframe
from splitters import SplitConfig, split_by_movie_id
from subsample import stratified_sample_by_label
from head_tail import apply_head_tail_truncation

paths = get_paths(ROOT) 

c:\Users\cola0\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df_raw = load_raw_imdb_spoiler_json(paths.data_raw)
df_all, _ = prepare_reviews_dataframe(df_raw)
print(f"Starting Dataset Length: {len(df_all)}")

Starting Dataset Length: 573913


In [3]:
split_cfg = SplitConfig(train_size=0.8, val_size=0.1, test_size=0.1, seed=42)
train_full, val_full, test_full = split_by_movie_id(df_all, split_cfg)

print(f"Split Iniziale: Train={len(train_full)}, Val={len(val_full)}, Test={len(test_full)}")

Split Iniziale: Train=467362, Val=52251, Test=54300


In [4]:
# Cella 3: Subsampling a 120k totali con Analisi Distribuzione
TARGET_TOTAL = 120_000
N_TRAIN = int(TARGET_TOTAL * 0.8)
N_VAL   = int(TARGET_TOTAL * 0.1)
N_TEST  = int(TARGET_TOTAL * 0.1)

# Applichiamo subsampling stratificato
train_120k = stratified_sample_by_label(train_full, N_TRAIN, seed=42)
val_120k   = stratified_sample_by_label(val_full, N_VAL, seed=42)
test_120k  = stratified_sample_by_label(test_full, N_TEST, seed=42)

# --- ANALISI DISTRIBUZIONE ---
def print_distribution(df, name):
    counts = df["label"].value_counts()
    props = df["label"].value_counts(normalize=True)
    
    print(f"\n📊 {name} Set ({len(df)} reviews):")
    print(f"   🟢 Non-Spoiler (0): {counts.get(0, 0)} ({props.get(0, 0):.2%})")
    print(f"   🔴 Spoiler     (1): {counts.get(1, 0)} ({props.get(1, 0):.2%})")
    
    # Calcolo del rapporto per pos_weight
    ratio = counts.get(0, 1) / max(1, counts.get(1, 1))
    print(f"   ⚖️ Ratio 0/1: {ratio:.2f} (Per ogni spoiler ci sono {ratio:.2f} non-spoiler)")
    return ratio

print("-" * 30)
r_train = print_distribution(train_120k, "TRAIN")
r_val   = print_distribution(val_120k, "VAL")
r_test  = print_distribution(test_120k, "TEST")
print("-" * 30)

# Salviamo il rapporto medio suggerito per il training
SUGGESTED_POS_WEIGHT = r_train
print(f"\n💡 SUGGESTION: Nel training, potresti impostare pos_weight = {SUGGESTED_POS_WEIGHT:.2f}")
print("   Questo dirà alla loss function di 'dare più importanza' agli spoiler rari.")

------------------------------

📊 TRAIN Set (96000 reviews):
   🟢 Non-Spoiler (0): 70755 (73.70%)
   🔴 Spoiler     (1): 25245 (26.30%)
   ⚖️ Ratio 0/1: 2.80 (Per ogni spoiler ci sono 2.80 non-spoiler)

📊 VAL Set (12000 reviews):
   🟢 Non-Spoiler (0): 8844 (73.70%)
   🔴 Spoiler     (1): 3156 (26.30%)
   ⚖️ Ratio 0/1: 2.80 (Per ogni spoiler ci sono 2.80 non-spoiler)

📊 TEST Set (12000 reviews):
   🟢 Non-Spoiler (0): 8844 (73.70%)
   🔴 Spoiler     (1): 3156 (26.30%)
   ⚖️ Ratio 0/1: 2.80 (Per ogni spoiler ci sono 2.80 non-spoiler)
------------------------------

💡 SUGGESTION: Nel training, potresti impostare pos_weight = 2.80
   Questo dirà alla loss function di 'dare più importanza' agli spoiler rari.


In [6]:
MODEL_NAME = "bert-large-uncased"

print("Processing Train...")
train_ht = apply_head_tail_truncation(train_120k, MODEL_NAME, max_length=512, head_len=128)

print("Processing Val...")
val_ht = apply_head_tail_truncation(val_120k, MODEL_NAME, max_length=512, head_len=128)

print("Processing Test...")
test_ht = apply_head_tail_truncation(test_120k, MODEL_NAME, max_length=512, head_len=128)


Processing Train...
Head-tail strategy: Head=128, Tail=382, total= 512


Token indices sequence length is longer than the specified maximum sequence length for this model (591 > 512). Running this sequence through the model will result in indexing errors
Applying Head+Tail: 100%|██████████| 96000/96000 [00:00<00:00, 172788.57it/s]


Processing Val...
Head-tail strategy: Head=128, Tail=382, total= 512


Token indices sequence length is longer than the specified maximum sequence length for this model (683 > 512). Running this sequence through the model will result in indexing errors
Applying Head+Tail: 100%|██████████| 12000/12000 [00:00<00:00, 160559.30it/s]


Processing Test...
Head-tail strategy: Head=128, Tail=382, total= 512


Token indices sequence length is longer than the specified maximum sequence length for this model (985 > 512). Running this sequence through the model will result in indexing errors
Applying Head+Tail: 100%|██████████| 12000/12000 [00:00<00:00, 169653.62it/s]


In [ ]:
sample = train_ht.iloc[0]
print(f"Review ID: {sample['review_id']}")
print(f"Original Text Len: {len(sample['text'])}")
print(f"Input IDs Len: {len(sample['input_ids'])}")
print(f"Tokens decoded: {sample['input_ids'][:10]} ... {sample['input_ids'][-10:]}")

Review ID: 393736
Original Text Len: 800
Input IDs Len: 172
Tokens decoded: [101, 2023, 2143, 2003, 6581, 1012, 2017, 2442, 2156, 2009] ... [1996, 2567, 1997, 7980, 1012, 5379, 2123, 14270, 17465, 102]


In [8]:
train_ht.to_parquet(paths.data_processed / "train_120k_ht.parquet", index=False)
val_ht.to_parquet(paths.data_processed / "val_120k_ht.parquet", index=False)
test_ht.to_parquet(paths.data_processed / "test_120k_ht.parquet", index=False)

print("✅ Dati salvati in data/processed/ (suffisso _ht.parquet)")

✅ Dati salvati in data/processed/ (suffisso _ht.parquet)
